# 🔬 SciCLIP: LoRA-Adapted CLIP for Scientific Figure Retrieval

> **Runtime → Change runtime type → A100 GPU** for best performance (~2 hrs full training)  
> T4 also works but slower (~5 hrs)

This notebook walks through the full SciCLIP pipeline:
1. Install dependencies
2. Clone the repo & prepare data
3. Train LoRA-CLIP (or run a quick mini-test)
4. Evaluate: Vanilla CLIP vs LoRA-CLIP
5. Build FAISS index & run retrieval demo

## 0. Check GPU

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")

## 1. Install Dependencies

In [ ]:
!pip install -q \
    "transformers>=4.38.0,<4.46.0" \
    "peft>=0.9.0,<0.15.0" \
    "faiss-gpu" \
    "datasets>=2.18.0" \
    "gradio>=3.50.0" \
    "accelerate>=0.27.0" \
    "huggingface-hub>=0.23.2,<1.0" \
    wandb tqdm

print("✅ Dependencies installed")

## 2. Clone SciCLIP Repo

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/sciclip.git"  # ← 나중에 본인 repo URL로 교체

if not os.path.exists("sciclip"):
    !git clone {REPO_URL}

%cd sciclip
!ls

## 3. Download & Prepare SciCap Data

- **Full run**: `--max_samples 20000` (~20 min download)
- **Mini test**: `--max_samples 200` (~1 min, just to verify pipeline)

In [ ]:
# ✏️ Set this to 200 for a quick sanity check, 20000 for full training
MAX_SAMPLES = 200  # change to 20000 for full run

In [ ]:
!python data/prepare_scicap.py --max_samples {MAX_SAMPLES} --output_dir data/

# Check output
!echo "--- train ---" && wc -l data/scicap_train.jsonl
!echo "--- val ---"   && wc -l data/scicap_val.jsonl

## 4. Train LoRA-CLIP

Key flags:
| Flag | Default | Description |
|------|---------|-------------|
| `--lora_r` | 8 | LoRA rank (ablation: 4, 8, 16) |
| `--epochs` | 5 | Training epochs |
| `--batch_size` | 64 | Batch size |
| `--wandb` | off | Enable W&B logging |
| `--ablation` | off | Run r=4,8,16 sequentially |

In [ ]:
# ✏️ Training config — adjust as needed
LORA_R      = 8
EPOCHS      = 5       # set to 1 for mini test
BATCH_SIZE  = 64
USE_WANDB   = False   # set True if you have a W&B account

In [ ]:
wandb_flag = "--wandb" if USE_WANDB else ""

!python train.py \
    --lora_r {LORA_R} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    {wandb_flag}

In [ ]:
# Verify checkpoint was saved
!ls checkpoints/lora_r{LORA_R}/best_adapter/
!cat checkpoints/lora_r{LORA_R}/history.json

## 5. Ablation Study (LoRA Rank r = 4, 8, 16)

Skip this cell if you only want the r=8 result.

In [ ]:
# ⚠️  This runs 3× training — takes ~6 hrs on A100 with 20k samples
# Uncomment to run:

# !python train.py --ablation --epochs {EPOCHS} --batch_size {BATCH_SIZE}

## 6. Evaluate: Vanilla CLIP vs LoRA-CLIP

In [ ]:
!python evaluate.py \
    --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
    --val_path data/scicap_val.jsonl

In [ ]:
# Or run in-notebook for inline results
import sys, torch
sys.path.insert(0, '.')

from evaluate import compare_baseline_vs_lora

results = compare_baseline_vs_lora(
    adapter_path=f"checkpoints/lora_r{LORA_R}/best_adapter",
    val_path="data/scicap_val.jsonl",
)

## 7. Build FAISS Retrieval Index

In [ ]:
!python build_index.py \
    --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
    --data_path data/scicap_train.jsonl \
    --index_path data/sciclip.index

!ls -lh data/sciclip.index data/index_metadata.json

## 8. CLI Retrieval Test

In [ ]:
# Test a few queries
queries = [
    "attention mechanism architecture diagram",
    "loss curve showing training convergence",
    "bar chart comparing model performance",
    "transformer encoder decoder architecture",
]

for q in queries:
    print(f"\n🔍 Query: '{q}'")
    !python retrieve.py \
        --query "{q}" \
        --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
        --index_path data/sciclip.index \
        --top_k 3

## 9. Gradio Demo

Launches a web UI at the public Gradio link.  
Shows **side-by-side comparison**: Vanilla CLIP vs LoRA-CLIP results.

In [ ]:
!python app.py \
    --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
    --index_path data/sciclip.index \
    --share  # generates public URL

## 10. Save Adapter to HuggingFace Hub (optional)

Upload the trained LoRA adapter so others can use it directly.

In [ ]:
# ✏️ Fill in your HuggingFace username and repo name
HF_USERNAME  = "jkhyjkhy"
HF_REPO_NAME = "sciclip-lora-r8"

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(f"{HF_USERNAME}/{HF_REPO_NAME}", exist_ok=True)

api.upload_folder(
    folder_path=f"checkpoints/lora_r{LORA_R}/best_adapter",
    repo_id=f"{HF_USERNAME}/{HF_REPO_NAME}",
    repo_type="model",
)

print(f"✅ Adapter uploaded: https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}")

## 11. Results Summary

After training, copy your results into the table below for the term paper.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print(f"{'Model':<20} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print("-" * 60)

# Fill in after running evaluate.py
# Example structure — replace with your actual results:
table = [
    {"model": "Vanilla CLIP",     "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
    {"model": "LoRA-CLIP (r=4)",  "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
    {"model": "LoRA-CLIP (r=8)",  "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
    {"model": "LoRA-CLIP (r=16)", "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
]

for row in table:
    print(f"{row['model']:<20} {row['r1']:>8} {row['r5']:>8} {row['r10']:>8} {row['mrr']:>8}")

print("=" * 60)
print("\n→ Copy this into README.md table when results are ready!")